# Stage C - CNN Depth Ablation
Deep learning model for CIFAR-10. This notebook compares CNN depth and checks the effect of augmentation.

In [ ]:
# Portable project setup. Works after cloning GitHub repo or extracting the ZIP.
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
# If Colab opens the notebook from another location, uncomment and edit this line:
# PROJECT_ROOT = Path("/content/MLFinalProject")
sys.path.insert(0, str(PROJECT_ROOT))
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

import numpy as np, pandas as pd, json
import tensorflow as tf
import matplotlib.pyplot as plt
from src.data import get_image_arrays, scale_to_float, compute_channel_stats, normalize, set_seeds
from src.augment import build_augmentation_layers
from src.metrics import compute_metrics, classification_report_df, confusion_matrix_array
from src import viz
set_seeds(42)
print("TensorFlow:", tf.__version__)

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = get_image_arrays()
x_train_f = scale_to_float(x_train); x_val_f = scale_to_float(x_val); x_test_f = scale_to_float(x_test)
mean, std = compute_channel_stats(x_train_f)
x_train_n = normalize(x_train_f, mean, std); x_val_n = normalize(x_val_f, mean, std); x_test_n = normalize(x_test_f, mean, std)
BATCH = 128
ds_train = tf.data.Dataset.from_tensor_slices((x_train_n, y_train)).shuffle(10000, seed=42).batch(BATCH).prefetch(tf.data.AUTOTUNE)
ds_val = tf.data.Dataset.from_tensor_slices((x_val_n, y_val)).batch(BATCH).prefetch(tf.data.AUTOTUNE)
ds_test = tf.data.Dataset.from_tensor_slices((x_test_n, y_test)).batch(BATCH).prefetch(tf.data.AUTOTUNE)
print("Train/Val/Test:", len(x_train), len(x_val), len(x_test))

In [ ]:
def conv_block(filters, dropout_rate=0.25):
    return [
        tf.keras.layers.Conv2D(filters, 3, padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Conv2D(filters, 3, padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Dropout(dropout_rate),
    ]

def build_cnn(n_blocks, filter_sizes, use_augmentation=True, name="cnn"):
    layers = []
    if use_augmentation:
        layers.append(build_augmentation_layers())
    for i in range(n_blocks):
        layers.extend(conv_block(filter_sizes[i]))
    layers += [tf.keras.layers.GlobalAveragePooling2D(), tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(0.5), tf.keras.layers.Dense(10, activation="softmax")]
    model = tf.keras.Sequential(layers, name=name)
    model.build((None, 32, 32, 3))
    return model

CONFIGS = {
    "CNN-shallow": {"n_blocks": 2, "filter_sizes": [32, 64]},
    "CNN-medium": {"n_blocks": 4, "filter_sizes": [32, 64, 128, 128]},
    "CNN-deep": {"n_blocks": 6, "filter_sizes": [32, 64, 128, 128, 256, 256]},
}
for name, cfg in CONFIGS.items():
    m = build_cnn(**cfg, name=name)
    print(name, "params:", m.count_params())

In [ ]:
def train_cnn(model, epochs=40, name="cnn"):
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(1e-3, decay_steps=epochs * len(ds_train))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr_schedule), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(MODEL_DIR / f"{name}.keras"), monitor="val_accuracy", save_best_only=True),
    ]
    return model.fit(ds_train, validation_data=ds_val, epochs=epochs, callbacks=callbacks, verbose=1)

results, histories, trained_models = {}, {}, {}
for model_name, cfg in CONFIGS.items():
    print("=" * 60)
    print("Training", model_name)
    model = build_cnn(**cfg, use_augmentation=True, name=model_name)
    history = train_cnn(model, epochs=40, name=model_name)
    histories[model_name] = history.history
    trained_models[model_name] = model
    y_pred = model.predict(ds_test, verbose=0).argmax(axis=1)
    metrics = compute_metrics(y_test, y_pred, model_name=model_name)
    metrics["n_blocks"] = cfg["n_blocks"]; metrics["n_params"] = model.count_params()
    results[model_name] = metrics
    print(model_name, metrics)

In [ ]:
print("Training CNN-medium without augmentation")
model_noaug = build_cnn(**CONFIGS["CNN-medium"], use_augmentation=False, name="CNN-medium-noaug")
history_noaug = train_cnn(model_noaug, epochs=40, name="CNN-medium-noaug")
y_pred_noaug = model_noaug.predict(ds_test, verbose=0).argmax(axis=1)
metrics_noaug = compute_metrics(y_test, y_pred_noaug, model_name="CNN-medium-noaug")
metrics_noaug["n_blocks"] = 4; metrics_noaug["n_params"] = model_noaug.count_params()
results["CNN-medium-noaug"] = metrics_noaug
histories["CNN-medium-noaug"] = history_noaug.history
print(metrics_noaug)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in ["CNN-shallow", "CNN-medium", "CNN-deep"]:
    h = histories[name]
    axes[0].plot(h["loss"], label=f"{name} train")
    axes[0].plot(h["val_loss"], linestyle="--", label=f"{name} val")
    axes[1].plot(h["accuracy"], label=f"{name} train")
    axes[1].plot(h["val_accuracy"], linestyle="--", label=f"{name} val")
axes[0].set_title("CNN loss curves"); axes[1].set_title("CNN accuracy curves")
for ax in axes: ax.set_xlabel("Epoch"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "stageC_depth_curves.png", dpi=150); plt.show()

# Depth vs accuracy
depths = [2, 4, 6]
accs = [results["CNN-shallow"]["accuracy"], results["CNN-medium"]["accuracy"], results["CNN-deep"]["accuracy"]]
viz.plot_depth_vs_accuracy(depths, accs)

best_name = max(["CNN-shallow", "CNN-medium", "CNN-deep"], key=lambda n: results[n]["accuracy"])
best_model = trained_models[best_name]
y_pred_best = best_model.predict(ds_test, verbose=0).argmax(axis=1)
cm = confusion_matrix_array(y_test, y_pred_best)
viz.plot_confusion_matrix(cm, title=f"Stage C {best_name} Confusion Matrix", save_name="stageC_cnn_confusion_matrix.png")

In [ ]:
cnn_results = []
for name, m in results.items():
    cnn_results.append({
        "model": name,
        "n_blocks": int(m.get("n_blocks", 4)),
        "n_params": int(m.get("n_params", 0)),
        "test_accuracy": float(m["accuracy"]),
        "macro_f1": float(m["macro_f1"]),
    })
results_df = pd.DataFrame(cnn_results)
display(results_df)
results_df.to_csv(TABLE_DIR / "stageC_cnn_results.csv", index=False)
with open(TABLE_DIR / "stageC_cnn_result.json", "w") as f: json.dump(cnn_results, f, indent=2)
print("Stage C CNN result saved.")

## Stage C interpretation
The CNN is the most suitable family for CIFAR-10 because it keeps the image as a spatial grid. Convolution uses local filters and weight sharing, so it can learn edges, textures, and object parts. Increasing depth improves the model until the gain becomes smaller, which shows the trade-off between performance, training time, and model size.